In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 📰 Freshness-Aware News RAG

## What We're Building

A news QA system that **prefers recent documents** over old ones and
**labels stale answers** when the best-matching docs are outdated.

## The Problem

```
Standard RAG:  "Who is the CEO of Twitter?"  → Might return: "Jack Dorsey" (2021 article)
Freshness RAG: "Who is the CEO of Twitter?"  → Returns: "Elon Musk" (2024 article)
                                              → Or labels 2021 answer as STALE
```

## How Freshness Scoring Works

```
final_score = semantic_similarity * (1 - α) + freshness_score * α

where:
  freshness_score = 1.0 - (days_old / max_age_days)  (clamped to [0, 1])
  α = freshness_weight (0.0 = ignore age, 1.0 = only age matters)
```

## Stack
- **ChromaDB** — vector store with metadata filtering
- **LangChain + Ollama** — LLM pipeline
- **datetime** — timestamp-based scoring

## Step 1 — Imports & Timestamped News Corpus

In [ ]:
from datetime import datetime, timedelta
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

In [ ]:
# Simulated news articles with dates — notice conflicting information across time
today = datetime.now()

news_articles = [
    # AI / LLM news — evolving over time
    Document(
        page_content="OpenAI released GPT-3 with 175 billion parameters, marking a breakthrough in large language models. The model can write essays, code, and poetry with remarkable quality.",
        metadata={"date": "2020-06-11", "source": "TechCrunch", "topic": "AI"}
    ),
    Document(
        page_content="OpenAI launched ChatGPT based on GPT-3.5 in November 2022. The chatbot reached 100 million users in two months, making it the fastest-growing consumer app in history.",
        metadata={"date": "2023-01-15", "source": "Reuters", "topic": "AI"}
    ),
    Document(
        page_content="OpenAI released GPT-4 in March 2023, a multimodal model that accepts images and text. It scores in the 90th percentile on the bar exam and significantly outperforms GPT-3.5.",
        metadata={"date": "2023-03-14", "source": "The Verge", "topic": "AI"}
    ),
    Document(
        page_content="OpenAI launched GPT-4o ('omni') in May 2024, a natively multimodal model that processes text, audio, and vision in real-time. Response latency dropped to 232ms average. The model is available free to all ChatGPT users.",
        metadata={"date": "2024-05-13", "source": "OpenAI Blog", "topic": "AI"}
    ),

    # Elon Musk / Twitter / X
    Document(
        page_content="Twitter banned Donald Trump from the platform permanently, citing 'risk of further incitement of violence.' CEO Jack Dorsey said it was the right decision but set a dangerous precedent.",
        metadata={"date": "2021-01-08", "source": "CNN", "topic": "social media"}
    ),
    Document(
        page_content="Elon Musk completed his $44 billion acquisition of Twitter in October 2022. He immediately fired CEO Parag Agrawal and other top executives. Musk declared himself 'Chief Twit'.",
        metadata={"date": "2022-10-27", "source": "CNBC", "topic": "social media"}
    ),
    Document(
        page_content="Elon Musk rebranded Twitter to X in July 2023, replacing the iconic blue bird logo. The company aims to become an 'everything app' for payments, messaging, and social media.",
        metadata={"date": "2023-07-23", "source": "BBC", "topic": "social media"}
    ),
    Document(
        page_content="Linda Yaccarino was appointed as CEO of X (formerly Twitter) in June 2023, while Elon Musk retained the role of Executive Chairman and CTO. Yaccarino, a former NBCUniversal executive, focuses on advertising partnerships.",
        metadata={"date": "2023-06-05", "source": "WSJ", "topic": "social media"}
    ),

    # Electric Vehicles
    Document(
        page_content="Tesla delivered 936,000 vehicles in 2021, a 87% increase from 2020. The Model 3 and Model Y accounted for 97% of deliveries. Tesla's market cap reached $1 trillion.",
        metadata={"date": "2022-01-03", "source": "Bloomberg", "topic": "EVs"}
    ),
    Document(
        page_content="Tesla delivered 1.81 million vehicles in 2023, but growth slowed to 38%. BYD overtook Tesla as the world's largest EV maker by volume in Q4 2023, selling 526,000 EVs vs Tesla's 484,000.",
        metadata={"date": "2024-01-05", "source": "Bloomberg", "topic": "EVs"}
    ),

    # Space
    Document(
        page_content="SpaceX launched Crew Dragon to the ISS in May 2020, the first crewed orbital spaceflight from US soil since the Space Shuttle retired in 2011. Astronauts Bob Behnken and Doug Hurley flew the mission.",
        metadata={"date": "2020-05-30", "source": "NASA", "topic": "space"}
    ),
    Document(
        page_content="SpaceX's Starship completed its first successful orbital test flight in June 2024, with both the Super Heavy booster and Starship upper stage landing successfully. It is the largest rocket ever built at 121 meters tall.",
        metadata={"date": "2024-06-15", "source": "SpaceNews", "topic": "space"}
    ),
]

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(news_articles, embeddings, collection_name="freshness")

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

print(f"Indexed {len(news_articles)} articles spanning {news_articles[0].metadata['date']} to {news_articles[-1].metadata['date']}")

## Step 2 — Standard RAG (No Freshness Awareness)

First, see what happens without freshness scoring.

In [ ]:
def standard_rag(question: str, k: int = 3) -> str:
    """Standard RAG — purely semantic similarity, ignores dates."""
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(f"[{d.metadata['date']} | {d.metadata['source']}] {d.page_content}" for d in docs)

    prompt = ChatPromptTemplate.from_template(
        "Answer based on the context. Mention dates when relevant.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})

    print(f"📌 Standard RAG (no freshness):")
    print(f"  Sources used: {[d.metadata['date'] for d in docs]}")
    print(f"  Answer: {answer}")
    return answer


# This might give an outdated answer!
standard_rag("What is OpenAI's latest model?")

## Step 3 — Freshness-Aware Retrieval

Combine semantic similarity with a date-based freshness score.

In [ ]:
def compute_freshness_score(doc_date_str: str, max_age_days: int = 1825) -> float:
    """Score from 0.0 (very old) to 1.0 (very recent).
    max_age_days=1825 means documents older than 5 years get score 0."""
    doc_date = datetime.strptime(doc_date_str, "%Y-%m-%d")
    days_old = (today - doc_date).days
    score = max(0.0, 1.0 - (days_old / max_age_days))
    return round(score, 3)


def freshness_aware_search(
    question: str,
    k_initial: int = 8,
    k_final: int = 3,
    freshness_weight: float = 0.3
) -> list[tuple[Document, float, float, float]]:
    """Retrieve docs, re-score with freshness, return top-k.

    Returns: [(doc, final_score, semantic_score, freshness_score), ...]
    """
    # Get more candidates than needed
    results = vectorstore.similarity_search_with_relevance_scores(question, k=k_initial)

    scored = []
    for doc, semantic_score in results:
        freshness = compute_freshness_score(doc.metadata["date"])
        final = semantic_score * (1 - freshness_weight) + freshness * freshness_weight
        scored.append((doc, final, semantic_score, freshness))

    # Sort by final score
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k_final]


# Visualize scoring
question = "What is OpenAI's latest model?"
results = freshness_aware_search(question, freshness_weight=0.3)

print(f"Query: {question}\n")
print(f"{'Date':<12} {'Semantic':>8} {'Fresh':>7} {'Final':>7}  Preview")
print("-" * 75)
for doc, final, sem, fresh in results:
    preview = doc.page_content[:50].replace('\n', ' ')
    print(f"{doc.metadata['date']:<12} {sem:>8.3f} {fresh:>7.3f} {final:>7.3f}  {preview}...")

## Step 4 — Staleness Detection

If the best-matching documents are old, warn the user.

In [ ]:
def detect_staleness(docs: list[tuple], stale_threshold_days: int = 365) -> dict:
    """Analyze if retrieved docs might be stale."""
    dates = []
    for doc, _, _, _ in docs:
        doc_date = datetime.strptime(doc.metadata["date"], "%Y-%m-%d")
        days_old = (today - doc_date).days
        dates.append(days_old)

    avg_age_days = sum(dates) / len(dates)
    newest_days = min(dates)
    oldest_days = max(dates)

    is_stale = newest_days > stale_threshold_days
    has_mixed_ages = (oldest_days - newest_days) > 365  # >1 year spread

    return {
        "is_stale": is_stale,
        "has_mixed_ages": has_mixed_ages,
        "avg_age_days": int(avg_age_days),
        "newest_days": newest_days,
        "oldest_days": oldest_days,
        "warning": (
            f"⚠️ STALE: Best sources are {newest_days}+ days old. Information may be outdated."
            if is_stale
            else (
                f"⚠️ MIXED AGES: Sources span {oldest_days - newest_days} days. Check for contradictions."
                if has_mixed_ages
                else "✅ Sources are recent."
            )
        )
    }


# Test staleness detection
for q in ["What is OpenAI's latest model?", "When did SpaceX first send astronauts?"]:
    results = freshness_aware_search(q)
    staleness = detect_staleness(results)
    print(f"Q: {q}")
    print(f"  {staleness['warning']}")
    print(f"  Newest: {staleness['newest_days']}d ago, Oldest: {staleness['oldest_days']}d ago\n")

## Step 5 — Full Freshness-Aware RAG Pipeline

In [ ]:
answer_prompt = ChatPromptTemplate.from_template(
    """Answer the question using the context below.
Each source has a date — prefer the most recent information.
If sources conflict, mention the discrepancy and go with the newest.

Context:
{context}

Freshness note: {freshness_note}

Question: {question}

Answer (mention dates when relevant):"""
)


def freshness_rag(question: str, freshness_weight: float = 0.3) -> str:
    """Full freshness-aware RAG pipeline."""
    print(f"\n{'='*60}")
    print(f"❓ {question}")
    print("=" * 60)

    # Freshness-aware retrieval
    results = freshness_aware_search(question, freshness_weight=freshness_weight)

    # Print scoring
    print(f"\n📊 Retrieval scores (freshness_weight={freshness_weight}):")
    for doc, final, sem, fresh in results:
        print(f"  [{doc.metadata['date']}] sem={sem:.3f} fresh={fresh:.3f} final={final:.3f}")

    # Staleness check
    staleness = detect_staleness(results)
    print(f"\n{staleness['warning']}")

    # Build context
    context = "\n\n".join(
        f"[{d.metadata['date']} | {d.metadata['source']}] {d.page_content}"
        for d, _, _, _ in results
    )

    chain = answer_prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "context": context,
        "question": question,
        "freshness_note": staleness["warning"]
    })

    print(f"\n📝 Answer: {answer}")
    return answer

In [ ]:
# Should prefer GPT-4o (2024) over GPT-3 (2020)
_ = freshness_rag("What is OpenAI's latest model?")

In [ ]:
# Should reflect the rebranding to X and current CEO
_ = freshness_rag("Who runs Twitter now?")

In [ ]:
# Should show BYD overtaking Tesla (2024) rather than just Tesla's 2021 numbers
_ = freshness_rag("Who is the world's largest EV manufacturer?")

## Step 6 — Effect of Freshness Weight

Compare how different weights change retrieval behavior.

In [ ]:
question = "Tell me about SpaceX missions"
print(f"Query: {question}\n")

for weight in [0.0, 0.3, 0.6, 0.9]:
    results = freshness_aware_search(question, freshness_weight=weight)
    dates = [d.metadata['date'] for d, _, _, _ in results]
    print(f"  weight={weight:.1f} → sources: {dates}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Freshness Score** | Linear decay: `1.0 - (days_old / max_age)` |
| **Score Fusion** | `semantic × (1-α) + freshness × α` |
| **Staleness Detection** | Flag when best docs are older than threshold |
| **Conflict Resolution** | Prefer newest when sources disagree |

## Tuning Guide

| Use Case | `freshness_weight` | Why |
|----------|-------------------|-----|
| Historical research | 0.0-0.1 | Age doesn't matter |
| General knowledge | 0.2-0.3 | Slight recency bias |
| News / current events | 0.4-0.6 | Strong recency preference |
| Stock market / live data | 0.7-0.9 | Only latest matters |

## 🔧 Extensions

- **Exponential decay**: Faster falloff for very recent content
- **Topic-aware freshness**: Some topics age faster (tech) vs slower (history)
- **Auto-refresh**: Detect stale answers and trigger web search for updates
- **Time-windowed retrieval**: ChromaDB metadata filter by date range